# Speech Graph — Visualization Notebook

This notebook constructs and analyzes **Speech Graphs** from transcribed verbal descriptions. Each verbatim is tokenized by whitespace, and consecutive word pairs form directed edges in a graph. The resulting metrics characterize the structural properties of the speaker's speech.

## Table of Contents
1. [Setup](#1-setup)
2. [LSCC Illustration](#2-lscc-illustration)
3. [Speech Graph Construction Example](#3-speech-graph-construction-example)
4. [Example on Shortest Verbatim](#4-example-on-shortest-verbatim)
5. [Tokenization Example](#5-tokenization-example)
6. [Full Dataset Processing](#6-full-dataset-processing)

---

## 1. Setup

Import libraries and load the dataset.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_INPUT  = '../SpeechGraph/data_SpeechGraph.csv'
DATA_OUTPUT = '../SpeechGraph/data_SpeechGraph2.csv'
#RESULTS_DIR = '../resultats'

data = pd.read_csv(DATA_INPUT, sep=';')
print(data.shape)
data.head()

---

## 2. LSCC Illustration

Schematic diagram of a directed speech graph highlighting the **Largest Strongly Connected Component (LSCC)** — the maximal subset of nodes mutually reachable by following directed edges. The orange box encloses the LSCC. This figure is saved as a standalone PNG for use in the thesis.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.set_aspect('equal')
ax.axis('off')

# ── Layout ────────────────────────────────────────────────────────────────────
N        = 13       # total nodes
SPACING  = 1.7      # distance between node centres
R        = 0.40     # node radius
Y        = 0.0
lcc_lo   = 3        # first node inside LCC (0-indexed)
lcc_hi   = 11       # last  node inside LCC (inclusive)

xs = [i * SPACING for i in range(N)]

# ── LCC highlight box ────────────────────────────────────────────────────────
PAD = 0.58
box = mpatches.FancyBboxPatch(
    (xs[lcc_lo] - R - PAD,  Y - R - PAD),
    (xs[lcc_hi] - xs[lcc_lo]) + 2 * (R + PAD),
    2 * (R + PAD),
    boxstyle="round,pad=0.15",
    linewidth=2.8, edgecolor='#E07040',
    facecolor='none', zorder=1,
)
ax.add_patch(box)

# ── Arrow helper ─────────────────────────────────────────────────────────────
SH = 19

def draw_arr(x1, y1, x2, y2, rad=0.0):
    ax.annotate(
        '', xy=(x2, y2), xytext=(x1, y1), zorder=2,
        arrowprops=dict(
            arrowstyle='-|>', color='black', lw=1.6,
            mutation_scale=14,
            shrinkA=SH, shrinkB=SH,
            connectionstyle=f'arc3,rad={rad}',
        ),
    )

# ── Forward chain ─────────────────────────────────────────────────────────────
for i in range(N - 1):
    draw_arr(xs[i], Y, xs[i + 1], Y)

# ── Backward arcs inside LCC ─────────────────────────────────────────────────
back_edges = [(6, 3), (7, 4), (8, 5), (10, 4), (11, 6)]
for src, dst in back_edges:
    span = src - dst
    rad  = 0.14 + span * 0.055
    draw_arr(xs[src], Y, xs[dst], Y, rad=rad)

# ── Nodes ──────────────────────────────────────────────────────────────────────
for x in xs:
    ax.add_patch(plt.Circle(
        (x, Y), R,
        color='#FFC107', ec='#CC9900', lw=1.8, zorder=3,
    ))

# ── Axis bounds ───────────────────────────────────────────────────────────────
ax.set_xlim(xs[0]  - R - 0.9, xs[-1] + R + 0.9)
ax.set_ylim(Y - R - 1.6,       Y + R + 1.0)

plt.tight_layout(pad=0)
plt.show()

---

## 3. Speech Graph Construction Example

Step-by-step construction of a speech graph from the sentence *"We see a woman from the back, the women is looking at the sea"*. Repeated words (e.g. *woman / women*) collapse to the same node, producing backward and skip edges that reflect lexical repetition in the speech stream.

In [ ]:
# ── Sentence and derived graph ────────────────────────────────────────────────
# "We see a woman from the back, the women is looking at the sea"
# Word sequence → unique node indices:
#   [0,1,2,3,4,5,6, 5,3, 7,8,9, 5,10]  (woman≈women merged)
# Consecutive pairs produce directed edges; repeated words fold back to existing nodes.

sentence  = '« We see a woman from the back, the women is looking at the sea »'
words     = ['we', 'see', 'a', 'wom-\nan', 'from', 'the', 'back', 'is', 'look-\ning', 'at', 'sea']
N         = len(words)   # 11 unique nodes

# Edge sets in terms of node indices in the linear layout above
seq_edges  = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(7,8),(8,9)]  # consecutive +1 steps
fwd_skip   = [(3,7),(5,10)]                                       # forward jumps (woman→is, the→sea)
back_edges = [(6,5),(5,3),(9,5)]                                  # backward (back→the, the→woman, at→the)

# ── Layout ────────────────────────────────────────────────────────────────────
SPACING = 1.8
R       = 0.42
Y       = 0.0
xs      = [i * SPACING for i in range(N)]
x_mid   = np.mean(xs)

fig, ax = plt.subplots(figsize=(18, 6.5))
ax.set_aspect('equal')
ax.axis('off')

# ── Sentence text ─────────────────────────────────────────────────────────────
ax.text(
    x_mid, Y + R + 3.5, sentence,
    ha='center', va='center',
    fontsize=13.5, color='#CC0000', fontstyle='italic',
)

# ── Big downward arrow ────────────────────────────────────────────────────────
ax.annotate(
    '',
    xy=(x_mid, Y + R + 1.75), xytext=(x_mid, Y + R + 2.95),
    arrowprops=dict(arrowstyle='-|>', color='black', lw=4, mutation_scale=30),
)

# ── Arrow helper ──────────────────────────────────────────────────────────────
# shrinkA/B ≈ node radius in pts  (18in fig, ~20 xlim units → ~65 pts/unit → R*65≈27 pts)
SH = 27

def arr(x1, y1, x2, y2, rad=0.0):
    ax.annotate(
        '', xy=(x2, y2), xytext=(x1, y1), zorder=2,
        arrowprops=dict(
            arrowstyle='-|>', color='black', lw=1.7,
            mutation_scale=14,
            shrinkA=SH, shrinkB=SH,
            connectionstyle=f'arc3,rad={rad}',
        ),
    )

# Sequential forward edges (straight)
for i, j in seq_edges:
    arr(xs[i], Y, xs[j], Y)

# Forward skip edges – arc above the node row (negative rad = curves upward)
for i, j in fwd_skip:
    span = j - i
    arr(xs[i], Y, xs[j], Y, rad=-(0.07 + span * 0.025))

# Backward edges – arc below the node row (positive rad = curves downward)
for i, j in back_edges:
    span = i - j
    arr(xs[i], Y, xs[j], Y, rad=0.18 + span * 0.055)

# ── Nodes (drawn last so they sit on top of arrows) ───────────────────────────
for k, x in enumerate(xs):
    ax.add_patch(plt.Circle((x, Y), R, color='#FFC107', ec='#CC9900', lw=1.8, zorder=3))
    ax.text(x, Y, words[k], ha='center', va='center',
            fontsize=7.5, fontweight='bold', zorder=4)

# ── Axis bounds ───────────────────────────────────────────────────────────────
ax.set_xlim(xs[0] - R - 0.7, xs[-1] + R + 0.7)
ax.set_ylim(Y - R - 2.3,      Y + R + 4.6)

plt.tight_layout(pad=0)
plt.show()

---

## 4. Example on Shortest Verbatim

Build and visualize the speech graph for the verbatim with the fewest tokens in the dataset, then compute all six retained metrics. This serves as a sanity check on the graph construction and metric pipeline before running over the full dataset.

In [ ]:
# Select the Verbatim SpeechGraph with the least tokens, ignoring empty entries
token_counts = data['Verbatim_SpeechGraph'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
valid_mask   = token_counts > 0

if not valid_mask.any():
    print("No valid Verbatim SpeechGraph found. Please check the data.")
else:
    min_tokens_row = data.loc[token_counts[valid_mask].idxmin()]
    verbatim = str(min_tokens_row['Verbatim_SpeechGraph'])
    print(f"Selected Verbatim with least tokens: {verbatim}")

    tokens = verbatim.split()
    G = nx.DiGraph()
    for i in range(len(tokens) - 1):
        G.add_edge(tokens[i], tokens[i + 1])

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=500, font_size=10)
    plt.title('SpeechGraph Visualization')
    plt.show()

    num_tokens = len(tokens)
    num_edges = G.number_of_edges()
    num_nodes = G.number_of_nodes()

    if len(G) > 0:
        LSCC = len(max(nx.strongly_connected_components(G), key=len))
        AD = num_edges / num_nodes if num_nodes > 0 else 0

        if nx.is_strongly_connected(G):
            subgraph = G
            ASPL = nx.average_shortest_path_length(G)
        else:
            largest_scc = max(nx.strongly_connected_components(G), key=len)
            subgraph = G.subgraph(largest_scc)
            ASPL = nx.average_shortest_path_length(subgraph) if len(subgraph) > 1 else None

        plt.figure(figsize=(10, 8))
        pos_lscc = nx.spring_layout(subgraph, seed=42)
        nx.draw(subgraph, pos_lscc, with_labels=True, node_color='lightgreen', edge_color='blue', node_size=500, font_size=10)
        plt.title('Largest Strongly Connected Component (LSCC) Visualization')
        plt.show()

        print(f'n_tokens: {num_tokens}')
        print(f'n_nodes:  {num_nodes}')
        print(f'n_edges:  {num_edges}')
        print(f'LSCC:     {LSCC}')
        print(f'ASPL:     {ASPL}')
        print(f'AD:       {AD:.4f}')
    else:
        print('Graph is empty. Metrics cannot be calculated.')

---

## 5. Tokenization Example

Illustrates what the tokenization step produces for the first verbatim in the dataset: the raw text string is split on whitespace, yielding a flat list of tokens that will form the node sequence of the speech graph.

In [ ]:
# Tokenization example: show how a verbatim is split into tokens
verbatim_example = data['Verbatim_SpeechGraph'][0]
print(f"Selected Verbatim: {verbatim_example}")

tokens_example = verbatim_example.split()
print(f"Tokens: {tokens_example}")

---

## 6. Full Dataset Processing

Iterate over all verbatims, build a directed speech graph for each, and compute the six metrics. Results are saved incrementally to `data_SpeechGraph2.csv` after each row so that progress is preserved if the loop is interrupted. A descriptive summary is printed at the end.

In [ ]:
# Metrics retained for analysis:
#   n_tokens   – raw speech length; controls for verbosity when comparing graph metrics across participants
#   n_nodes    – unique words used; proxy for lexical diversity
#   n_edges    – unique word transitions; captures syntactic/semantic connectivity beyond vocabulary size
#   LSCC       – size of the Largest Strongly Connected Component; reflects narrative cohesion
#                (how many words are reachable from one another following the directed speech flow)
#   ASPL       – Average Shortest Path Length within the LSCC
#                computed on the full graph when strongly connected, otherwise on the LSCC subgraph
#   AD         – Average Degree (edges / nodes)

for col in ['n_tokens', 'n_nodes', 'n_edges', 'LSCC', 'ASPL', 'AD']:
    data[col] = 0.0

for index, row in tqdm(data.iterrows(), total=len(data), desc='Processing verbatims'):
    verbatim = str(row['Verbatim_SpeechGraph'])
    if not pd.notna(row['Verbatim_SpeechGraph']) or verbatim.strip() == '':
        continue

    tokens = verbatim.split()
    G = nx.DiGraph()
    for i in range(len(tokens) - 1):
        G.add_edge(tokens[i], tokens[i + 1])

    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()

    data.at[index, 'n_tokens'] = len(tokens)
    data.at[index, 'n_nodes'] = num_nodes
    data.at[index, 'n_edges'] = num_edges

    lscc_size = len(max(nx.strongly_connected_components(G), key=len))
    data.at[index, 'LSCC'] = lscc_size
    data.at[index, 'AD'] = num_edges / num_nodes if num_nodes > 0 else 0

    if nx.is_strongly_connected(G):
        data.at[index, 'ASPL'] = nx.average_shortest_path_length(G)
    else:
        # Use LSCC subgraph: ASPL is only defined on strongly connected graphs
        largest_scc = max(nx.strongly_connected_components(G), key=len)
        sg = G.subgraph(largest_scc)
        data.at[index, 'ASPL'] = nx.average_shortest_path_length(sg) if len(sg) > 1 else 0

    data.to_csv(DATA_OUTPUT, sep=';', index=False)

data.to_csv(DATA_OUTPUT, sep=';', index=False)
print('Done.')
print(data[['n_tokens', 'n_nodes', 'n_edges', 'LSCC', 'ASPL', 'AD']].describe())